# ⚓ Análise de Custos Portuários — Porto de Santos
> **Simulação e identificação de gargalos financeiros em operações de Importação**

---

## 💼 Contexto de Negócio

Em operações de comércio exterior, a **previsibilidade de custos** é um dos maiores desafios enfrentados por importadores. A complexidade das tarifas portuárias — que misturam valores fixos e percentuais — dificulta saber com antecedência o custo real de armazenagem de uma carga.

A regra tarifária do Porto de Santos para container 40' segue a lógica:

$$\text{Custo} = \max(\text{R\$ 781,71},\ \text{Valor CIF} \times 0{,}20\%)$$

**Objetivo deste notebook:** Automatizar a decisão tarifária, identificar o ponto de breakeven e gerar visualizações estratégicas para suporte à decisão.

---

## 1. Imports e Configuração

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import Patch

# Reprodutibilidade
np.random.seed(42)

# ── Parâmetros tarifários ──────────────────────────────────────────
VALOR_MINIMO_ARMAZENAGEM = 781.71   # Tarifa mínima (floor) — Container 40'
TAXA_AD_VALOREM          = 0.0020   # 0,20% sobre o valor CIF

# ── Parâmetros da simulação ────────────────────────────────────────
VALOR_MIN_CIF = 200_000
VALOR_MAX_CIF = 600_000

# ── Paleta de cores ────────────────────────────────────────────────
COR_AD_VALOREM   = "#3498DB"
COR_VALOR_MINIMO = "#E67E22"
COR_ACUMULADO    = "#C0392B"

MESES = ["Jan","Fev","Mar","Abr","Mai","Jun",
         "Jul","Ago","Set","Out","Nov","Dez"]

print("✅ Configuração carregada com sucesso")

---

## 2. Funções Core

A função abaixo encapsula a regra de negócio central do projeto. Separar a lógica em funções reutilizáveis é uma boa prática — facilita testes, manutenção e reaproveitamento.

In [ ]:
def custo_armazenagem(valor_cif: pd.Series) -> pd.Series:
    """Aplica a regra tarifária: Custo = max(Mínimo, CIF × 0,20%)"""
    return np.maximum(VALOR_MINIMO_ARMAZENAGEM, valor_cif * TAXA_AD_VALOREM).round(2)

def formatar_reais(valor: float) -> str:
    """Formata valor em R$ no padrão brasileiro."""
    return f"R$ {valor:,.0f}".replace(",", ".")

# Ponto de breakeven matemático: CIF onde Ad Valorem = Mínimo
BREAKEVEN_CIF = VALOR_MINIMO_ARMAZENAGEM / TAXA_AD_VALOREM

print(f"📌 Ponto de Breakeven: {formatar_reais(BREAKEVEN_CIF)}")
print(f"   → Abaixo desse valor: cobra-se o mínimo de R$ {VALOR_MINIMO_ARMAZENAGEM:,.2f}")
print(f"   → Acima desse valor: cobra-se 0,20% sobre o CIF")

---

## 3. Análise de Breakeven (Threshold Analysis)

Esta análise responde à pergunta mais crítica para o importador:

> *"A partir de qual valor de carga começo a pagar proporcionalmente ao que importo?"*

- **Zona de Ineficiência (laranja):** Cargas abaixo do breakeven pagam uma taxa efetiva *superior* a 0,20%, penalizando importadores de produtos de baixo valor.
- **Zona de Risco (azul):** Acima do breakeven, o custo escala com o valor CIF e a variação cambial passa a impactar diretamente o custo de armazenagem.

In [ ]:
# Gera um range de valores CIF para a curva
cif_range = np.linspace(100_000, 1_000_000, 500)
custos    = np.maximum(VALOR_MINIMO_ARMAZENAGEM, cif_range * TAXA_AD_VALOREM)

fig, ax = plt.subplots(figsize=(12, 6))

# Curva principal
ax.plot(cif_range, custos, color="#2C3E50", linewidth=2.5, label="Custo de Armazenagem")

# Zonas coloridas
ax.fill_between(cif_range, custos,
                where=(cif_range <= BREAKEVEN_CIF),
                color=COR_VALOR_MINIMO, alpha=0.15, label="Zona de Ineficiência (Tarifa Mínima)")
ax.fill_between(cif_range, custos,
                where=(cif_range >= BREAKEVEN_CIF),
                color=COR_AD_VALOREM, alpha=0.15, label="Zona de Risco (Ad Valorem)")

# Linha vertical do breakeven
ax.axvline(x=BREAKEVEN_CIF, color="#E74C3C", linestyle="--", linewidth=1.8)
ax.axhline(y=VALOR_MINIMO_ARMAZENAGEM, color="gray", linestyle=":", linewidth=1.2)

# Anotação do breakeven
ax.annotate(
    f"Breakeven\n{formatar_reais(BREAKEVEN_CIF)}",
    xy=(BREAKEVEN_CIF, VALOR_MINIMO_ARMAZENAGEM),
    xytext=(BREAKEVEN_CIF + 60_000, VALOR_MINIMO_ARMAZENAGEM + 300),
    fontsize=10, fontweight="bold", color="#E74C3C",
    arrowprops=dict(arrowstyle="->", color="#E74C3C")
)

ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"R$ {v/1000:.0f}k"))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: formatar_reais(v)))
ax.set_xlabel("Valor CIF da Carga (R$)", fontsize=11)
ax.set_ylabel("Custo de Armazenagem (R$)", fontsize=11)
ax.set_title("Análise de Breakeven — Tarifa Mínima vs Ad Valorem", fontsize=14, fontweight="bold", pad=16)
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(alpha=0.3, linestyle="--")

plt.tight_layout()
plt.savefig("reports/images/breakeven_chart.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("✅ Gráfico salvo em reports/images/breakeven_chart.png")

---

## 4. Geração de Dados (Mock Data)

Simulamos um ano fiscal com valores CIF variando entre R$ 200k e R$ 600k, representando a realidade de um importador de médio porte no Porto de Santos.

In [ ]:
df = pd.DataFrame({
    "Mes":      MESES,
    "Valor_CIF": np.random.uniform(VALOR_MIN_CIF, VALOR_MAX_CIF, size=12).round(2)
})

df["Custo_Armazenagem"] = custo_armazenagem(df["Valor_CIF"])
df["Usou_Minimo"]       = df["Custo_Armazenagem"] == VALOR_MINIMO_ARMAZENAGEM
df["Taxa_Efetiva_%"]    = ((df["Custo_Armazenagem"] / df["Valor_CIF"]) * 100).round(4)
df["Acumulado"]         = df["Custo_Armazenagem"].cumsum()

print("📊 Dataset gerado com sucesso:\n")
print(df[["Mes","Valor_CIF","Custo_Armazenagem","Taxa_Efetiva_%","Usou_Minimo"]].to_string(index=False))

---

## 5. Simulação Anual — Combo Chart

O gráfico abaixo responde duas perguntas ao mesmo tempo:
1. **Barras:** *Quanto pagamos em cada mês?* — Cor laranja = mínimo ativado | Cor azul = Ad Valorem
2. **Linha vermelha:** *Como o custo acumula ao longo do ano?*

Meses com a barra laranja são **oportunidades de consolidação de carga** — o importador poderia combinar duas cargas menores em uma maior para diluir o custo fixo.

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 6))
x     = np.arange(len(df["Mes"]))
width = 0.6

cores_barras = [COR_VALOR_MINIMO if uso else COR_AD_VALOREM for uso in df["Usou_Minimo"]]
barras = ax1.bar(x, df["Custo_Armazenagem"], width, color=cores_barras,
                 edgecolor="white", linewidth=0.8)

ax1.set_xlabel("Mês", fontsize=11)
ax1.set_ylabel("Custo Mensal (R$)", fontsize=11, color="#2C3E50")
ax1.set_xticks(x)
ax1.set_xticklabels(df["Mes"])
ax1.yaxis.set_major_formatter(FuncFormatter(lambda v, _: formatar_reais(v)))
ax1.set_ylim(0, df["Custo_Armazenagem"].max() * 1.25)
ax1.grid(axis="y", alpha=0.3, linestyle="--")

for bar, valor in zip(barras, df["Custo_Armazenagem"]):
    ax1.annotate(formatar_reais(valor),
                 xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                 xytext=(0, 5), textcoords="offset points",
                 ha="center", va="bottom", fontsize=8, fontweight="bold", rotation=90)

ax2 = ax1.twinx()
ax2.plot(x, df["Acumulado"], color=COR_ACUMULADO, marker="o",
         markersize=7, linewidth=2.2, label="Custo Acumulado")
ax2.set_ylabel("Custo Acumulado (R$)", fontsize=11, color=COR_ACUMULADO)
ax2.tick_params(axis="y", labelcolor=COR_ACUMULADO)
ax2.yaxis.set_major_formatter(FuncFormatter(lambda v, _: formatar_reais(v)))
ax2.set_ylim(0, df["Acumulado"].max() * 1.15)

for xi, yi in zip(x, df["Acumulado"]):
    ax2.annotate(formatar_reais(yi), xy=(xi, yi), xytext=(0, 8),
                 textcoords="offset points", ha="center",
                 fontsize=8, color=COR_ACUMULADO, fontweight="bold")

ax1.legend(handles=[
    Patch(facecolor=COR_AD_VALOREM,   label="Ad Valorem (0,20% CIF)"),
    Patch(facecolor=COR_VALOR_MINIMO, label=f"Valor Mínimo (R$ 781,71) — ⚠️ Oportunidade de consolidação")
], loc="upper left", framealpha=0.9, fontsize=9)

plt.title("Simulação Anual: Custos Portuários por Mês + Acumulado",
          fontsize=14, fontweight="bold", pad=16)
fig.tight_layout()
plt.savefig("reports/images/monthly_analysis.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("✅ Gráfico salvo em reports/images/monthly_analysis.png")

---

## 6. Análise de Sensibilidade — "E se?"

O dólar é uma das maiores variáveis de risco em operações de importação. Um aumento cambial eleva o valor CIF em reais, podendo cruzar o ponto de breakeven e tornar o custo de armazenagem volátil.

Abaixo simulamos o impacto de variações cambiais de **-20% a +30%** sobre o custo total anual.

In [ ]:
variacoes     = np.arange(-0.20, 0.31, 0.05)
custos_totais = []

for var in variacoes:
    cif_ajustado    = df["Valor_CIF"] * (1 + var)
    custo_ajustado  = custo_armazenagem(cif_ajustado).sum()
    custos_totais.append(custo_ajustado)

custo_base = df["Custo_Armazenagem"].sum()

fig, ax = plt.subplots(figsize=(12, 5))
cores = [COR_VALOR_MINIMO if c < custo_base else COR_AD_VALOREM for c in custos_totais]
barras = ax.bar([f"{v*100:+.0f}%" for v in variacoes], custos_totais,
                color=cores, edgecolor="white")

ax.axhline(y=custo_base, color="#E74C3C", linestyle="--", linewidth=1.8,
           label=f"Custo base: {formatar_reais(custo_base)}")

for bar, valor in zip(barras, custos_totais):
    ax.annotate(formatar_reais(valor),
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 4), textcoords="offset points",
                ha="center", fontsize=8, fontweight="bold")

ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: formatar_reais(v)))
ax.set_xlabel("Variação Cambial (%)", fontsize=11)
ax.set_ylabel("Custo Total Anual (R$)", fontsize=11)
ax.set_title("Análise de Sensibilidade: Impacto da Variação Cambial no Custo Anual",
             fontsize=13, fontweight="bold", pad=14)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3, linestyle="--")

plt.tight_layout()
plt.savefig("reports/images/sensitivity_analysis.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("✅ Gráfico salvo em reports/images/sensitivity_analysis.png")

---

## 7. Resultado Financeiro

Resumo consolidado da simulação anual com cenários otimista, base e pessimista.

In [ ]:
total_anual       = df["Custo_Armazenagem"].sum()
meses_minimo      = df["Usou_Minimo"].sum()
meses_ad_valorem  = (~df["Usou_Minimo"]).sum()
taxa_media        = df["Taxa_Efetiva_%"].mean()

# Cenários com variação cambial de ±10%
cenario_pessimista = custo_armazenagem(df["Valor_CIF"] * 1.10).sum()
cenario_otimista   = custo_armazenagem(df["Valor_CIF"] * 0.90).sum()

print("=" * 60)
print("  RESULTADO FINANCEIRO — SIMULAÇÃO ANUAL")
print("=" * 60)
print(f"\n📦 Meses com Tarifa Mínima ativada : {meses_minimo}/12")
print(f"📈 Meses com Ad Valorem            : {meses_ad_valorem}/12")
print(f"📊 Taxa efetiva média              : {taxa_media:.4f}%")
print(f"\n💰 Custo Total Base                : {formatar_reais(total_anual)}")
print(f"🔴 Cenário Pessimista (+10% câmbio): {formatar_reais(cenario_pessimista)}")
print(f"🟢 Cenário Otimista  (-10% câmbio) : {formatar_reais(cenario_otimista)}")
print("=" * 60)

resumo = pd.DataFrame({
    "Cenário":       ["Otimista (-10%)", "Base", "Pessimista (+10%)"],
    "Custo Total":   [formatar_reais(cenario_otimista), formatar_reais(total_anual), formatar_reais(cenario_pessimista)],
    "Variação":      [f"{((cenario_otimista/total_anual)-1)*100:+.1f}%", "—", f"{((cenario_pessimista/total_anual)-1)*100:+.1f}%"]
})
print("\n", resumo.to_string(index=False))

---

## 8. Conclusões Estratégicas

### 📌 Principais Achados

1. **Ponto de Breakeven em R$ 390.855:** Cargas abaixo desse valor pagam uma taxa efetiva superior a 0,20%, penalizando importadores de produtos de baixo valor agregado.

2. **Cláusula Mínima como alavanca:** Meses em que o valor mínimo é ativado representam oportunidades concretas de **consolidação de carga** — combinar duas cargas menores em um único container pode reduzir o custo proporcionalmente.

3. **Exposição cambial:** Um aumento de 10% no dólar não impacta linearmente o custo — cargas próximas ao breakeven sofrem um salto de custo mais abrupto do que cargas que já operam no regime Ad Valorem.

### 🎯 Recomendações

| Perfil do Importador | Recomendação |
|---|---|
| Carga CIF < R$ 390k | Consolidar cargas para diluir o custo fixo |
| Carga CIF > R$ 390k | Monitorar variação cambial — priorizar hedge |
| Alta frequência de importação | Negociar tabela diferenciada com o operador portuário |

---
*Notebook desenvolvido por Gabriel Carvalho — [github.com/gbcarvalhoo](https://github.com/gbcarvalhoo)*